# From Hamming to Steane: the Week 2 bridge

This notebook documents the transition from the classical Hamming $[7,4,3]$ code to the Steane $[[7,1,3]]$ CSS code.

The goal is not only to run code, but to make the structure visible:

$$
\text{Hamming parity checks}
\rightarrow
\text{Pauli symplectic algebra}
\rightarrow
\text{stabilizers}
\rightarrow
\text{CSS checks}
\rightarrow
\text{Steane code}.
$$

By the end, we will verify the CSS commutation condition, show that the code encodes one logical qubit, validate logical operators, and exhaustively check all 21 single-qubit Pauli errors.

## 1. Imports

The notebook uses the tested functions in `src/softqec/` rather than reimplementing the algorithms here.

In [ ]:
import numpy as np

from softqec.css import css_checks_commute, css_syndrome, number_logical_qubits
from softqec.gf2 import matmul, rank
from softqec.pauli import commutes
from softqec.steane import (
    LOGICAL_X,
    LOGICAL_Z,
    N_PHYSICAL,
    STEANE_H,
    STEANE_H_X,
    STEANE_H_Z,
    correction_succeeds,
    decode_ideal_css,
    steane_parameters,
    steane_stabilizers,
)


## 2. The Hamming parity-check matrix

The Steane construction starts from a $3\times7$ Hamming parity-check matrix. Each column is a distinct nonzero three-bit vector, which is why a single-bit error has a unique syndrome.

In [ ]:
STEANE_H

In [ ]:
for qubit in range(N_PHYSICAL):
    print(f"qubit {qubit + 1}: syndrome {tuple(int(v) for v in STEANE_H[:, qubit])}")

### Classical-to-quantum dictionary

| Classical coding idea | CSS / stabilizer counterpart |
|---|---|
| parity-check matrix $H$ | quantum check matrices $H_X, H_Z$ |
| bit-error vector $e$ | Pauli components $(e_X\mid e_Z)$ |
| syndrome $He^T$ | $s_Z=H_Ze_X$, $s_X=H_Xe_Z$ |
| valid parity checks | mutually commuting stabilizers |
| correction of bit errors | correction modulo the stabilizer group |

This is the main conceptual bridge of the repository.

## 3. CSS construction

For the Steane code we take

$$H_X = H_Z = H.$$

The CSS commutation requirement is

$$H_XH_Z^T=0\pmod 2.$$

A zero entry means the corresponding X-type and Z-type stabilizer generators overlap on an even number of qubits, so they commute.

In [ ]:
commutation_matrix = matmul(STEANE_H_X, STEANE_H_Z.T)
commutation_matrix

In [ ]:
print("CSS checks commute:", css_checks_commute(STEANE_H_X, STEANE_H_Z))
print("rank(H_X):", rank(STEANE_H_X))
print("rank(H_Z):", rank(STEANE_H_Z))

## 4. Why the code encodes one logical qubit

For a CSS code with independent checks,

$$k=n-\operatorname{rank}(H_X)-\operatorname{rank}(H_Z).$$

For Steane,

$$k=7-3-3=1.$$

Therefore the construction defines a $[[7,1,3]]$ code.

In [ ]:
print("number of logical qubits:", number_logical_qubits(STEANE_H_X, STEANE_H_Z))
print("(n, k, d):", steane_parameters())

## 5. Six stabilizer generators

The first three generators are X-type and the last three are Z-type. The binary symplectic representation stores their X and Z components separately.

In [ ]:
stabilizer_x, stabilizer_z = steane_stabilizers()
print("X components:\n", stabilizer_x)
print("\nZ components:\n", stabilizer_z)

## 6. Logical operators

A convenient choice is

$$\bar X=X^{\otimes7},\qquad \bar Z=Z^{\otimes7}.$$

A valid logical operator must commute with every stabilizer but must not itself be a stabilizer. In addition, $\bar X$ and $\bar Z$ must anticommute.

In [ ]:
zero = np.zeros(N_PHYSICAL, dtype=np.uint8)

logical_x_commutes = all(
    commutes(LOGICAL_X, zero, stabilizer_x[i], stabilizer_z[i])
    for i in range(stabilizer_x.shape[0])
)

logical_z_commutes = all(
    commutes(zero, LOGICAL_Z, stabilizer_x[i], stabilizer_z[i])
    for i in range(stabilizer_x.shape[0])
)

logical_xz_commute = commutes(LOGICAL_X, zero, zero, LOGICAL_Z)

print("Xbar commutes with all stabilizers:", logical_x_commutes)
print("Zbar commutes with all stabilizers:", logical_z_commutes)
print("Xbar and Zbar commute:", logical_xz_commute)
print("Expected final line: False, because logical X and Z anticommute.")

## 7. Example syndromes

Z-type checks detect X errors, while X-type checks detect Z errors. A Y error contains both X and Z components and therefore activates both syndrome families.

In [ ]:
def single_qubit_error(qubit: int, pauli: str):
    ex = np.zeros(N_PHYSICAL, dtype=np.uint8)
    ez = np.zeros(N_PHYSICAL, dtype=np.uint8)

    if pauli in {"X", "Y"}:
        ex[qubit] = 1
    if pauli in {"Z", "Y"}:
        ez[qubit] = 1

    return ex, ez


for pauli in ["X", "Z", "Y"]:
    ex, ez = single_qubit_error(0, pauli)
    syndrome_x_checks, syndrome_z_checks = css_syndrome(
        ex, ez, STEANE_H_X, STEANE_H_Z
    )
    print(
        f"{pauli} on qubit 1 -> "
        f"X-check syndrome {tuple(syndrome_x_checks)}, "
        f"Z-check syndrome {tuple(syndrome_z_checks)}"
    )

## 8. Exhaustive validation of all 21 single-qubit Pauli errors

A distance-3 code must correct every weight-1 Pauli error. There are

$$7\times3=21$$

such errors: seven X errors, seven Z errors, and seven Y errors.

In [ ]:
results = []

for pauli in ["X", "Z", "Y"]:
    for qubit in range(N_PHYSICAL):
        ex, ez = single_qubit_error(qubit, pauli)
        cx, cz = decode_ideal_css(ex, ez)
        success = correction_succeeds(ex, ez, cx, cz)
        results.append((pauli, qubit + 1, success))

n_success = sum(success for _, _, success in results)

print(f"Corrected {n_success}/{len(results)} single-qubit Pauli errors")
assert n_success == 21

## 9. Why success is defined modulo stabilizers

In quantum error correction, the decoder does **not** need to reproduce the exact physical error. If the actual error is $E$ and the applied correction is $C$, decoding succeeds whenever

$$EC\in\mathcal S,$$

where $\mathcal S$ is the stabilizer group. Stabilizers act trivially on code states, so two physical errors that differ by a stabilizer have the same logical action.

This equivalence is the reason later soft decoding will aggregate probabilities over stabilizer cosets rather than selecting only one physical error pattern.

## 10. Week 2 checkpoint

At this stage the repository has established:

- binary symplectic Pauli representation;
- commutation and stabilizer syndromes;
- CSS check matrices and the condition $H_XH_Z^T=0$;
- the Steane $[[7,1,3]]$ construction;
- valid logical $\bar X$ and $\bar Z$ operators;
- ideal X/Z syndrome decoding;
- exhaustive correction of all 21 single-qubit Pauli errors.

Week 3 will keep this code fixed and replace ideal binary syndrome information with a noisy analog syndrome model, allowing hard-threshold and soft decoding to be compared.